# Claude Code as a headless worker

This notebook runs [Claude Code](https://code.claude.com/docs/en/overview) non-interactively in a prokube sandbox on a Git repository and shows its answer and the diff. The Anthropic API key never enters the sandbox; see the [README](./README.md) for how that works.

Sandboxes must be enabled on your prokube cluster, the Lab needs sandbox access, and you need an Anthropic API key.

## Setup

In [ ]:
%pip install -q "prokube-sdk @ git+https://github.com/prokube/prokube-sdk@main"

## The API key

The example needs your Anthropic API key, stored in a Kubernetes Secret in your workspace. The key never enters the sandbox: the egress gateway adds it to requests to `api.anthropic.com`.

Create the Secret once in the prokube UI:

1. Open the user menu (top right) and select **K8s Secrets**, then **Add Secret**.
2. Enter the name `anthropic-api-key` and add one entry: key `ANTHROPIC_API_KEY`, value your Anthropic API key.
3. Save.

Or from a terminal in this Lab:

```bash
read -rs KEY && kubectl create secret generic anthropic-api-key --from-literal=ANTHROPIC_API_KEY="$KEY"; unset KEY
```

The next cell checks that the Secret exists and adds the label `prokube.ai/sandbox-credential=true`, which lets the workspace's sandboxes use it.

In [ ]:
secret = !kubectl get secret anthropic-api-key -o name --ignore-not-found
if not secret:
    raise RuntimeError("No Secret anthropic-api-key yet. Create it as described above, then run this cell again.")
!kubectl label secret anthropic-api-key prokube.ai/sandbox-credential=true --overwrite > /dev/null
print("Found the Secret anthropic-api-key and made it available to sandboxes.")

## Create the sandbox

In [ ]:
from prokube.sandbox import SandboxClient

sandboxes = SandboxClient()  # the Lab's workspace and identity, no login
SANDBOX = "claude-code"

Create the sandbox with its own network rules instead of the preset's: the Anthropic API, GitHub and npm, nothing else. Claude Code sends a placeholder key, and the egress gateway replaces the `x-api-key` header with the key from the Secret.

In [ ]:
from prokube.sandbox import inject, passthrough

key = sandboxes.secret_uri("anthropic-api-key", "ANTHROPIC_API_KEY")
sandboxes.create(SANDBOX, network=[
    inject("api.anthropic.com", "x-api-key", key),
    passthrough("github.com", "registry.npmjs.org"),
]);

Install Claude Code into `/workspace` and clone a small repository:

In [ ]:
CLAUDE = "/workspace/claude-code/node_modules/.bin/claude"

sandboxes.run(
    SANDBOX,
    "npm install --silent --prefix /workspace/claude-code @anthropic-ai/claude-code"
    " && git clone -q --depth 1 https://github.com/pypa/sampleproject /workspace/sampleproject",
    timeout=600,
)

## Run Claude Code

`claude -p` runs one task and exits. The sandbox is the security boundary: it is isolated from the Lab and reaches only the Anthropic API, GitHub and npm. So Claude Code may edit files and run commands in it without asking (`--dangerously-skip-permissions`):

In [ ]:
import json
import shlex

TASK = "Add a function add_two to src/sample/simple.py, like add_one, and a test for it. Run the tests."

result = sandboxes.run(
    SANDBOX,
    f"{CLAUDE} -p {shlex.quote(TASK)} --output-format json --dangerously-skip-permissions",
    cwd="/workspace/sampleproject",
    env={"ANTHROPIC_API_KEY": "placeholder"},
    timeout=600,
)
print(json.loads(result.stdout)["result"])

The changes:

In [ ]:
print(sandboxes.run(SANDBOX, "git add -A && git diff --staged", cwd="/workspace/sampleproject").stdout)

## Clean up

Delete the sandbox. The README shows how to delete the Secret.

In [ ]:
sandboxes.delete(SANDBOX)
sandboxes.close()